# Formatting Utilities

> Utility functions for formatting file sizes, timestamps, and other display values.

In [ ]:
#| default_exp utils.formatting

In [ ]:
#| export
import fnmatch
import mimetypes
from datetime import datetime
from typing import List, Optional

In [ ]:
#| hide
from nbdev.showdoc import *

## File Size Formatting

In [ ]:
#| export
def format_file_size(
    size_bytes: int  # Size in bytes
) -> str:  # Human-readable size string (e.g., "15.2 MB")
    """Format file size in human-readable format."""
    if size_bytes < 0:
        return "0 B"
    
    units = ["B", "KB", "MB", "GB", "TB", "PB"]
    unit_index = 0
    size = float(size_bytes)
    
    while size >= 1024 and unit_index < len(units) - 1:
        size /= 1024
        unit_index += 1
    
    if unit_index == 0:
        return f"{int(size)} {units[unit_index]}"
    return f"{size:.1f} {units[unit_index]}"

In [ ]:
# Test format_file_size
assert format_file_size(0) == "0 B"
assert format_file_size(512) == "512 B"
assert format_file_size(1024) == "1.0 KB"
assert format_file_size(1536) == "1.5 KB"
assert format_file_size(1048576) == "1.0 MB"
assert format_file_size(1572864) == "1.5 MB"
assert format_file_size(1073741824) == "1.0 GB"
print("format_file_size tests passed!")

format_file_size tests passed!


## Timestamp Formatting

In [ ]:
#| export
def format_timestamp(
    timestamp: float  # Unix timestamp
) -> str:  # Human-readable date string
    """Format timestamp to human-readable date with relative time for recent files."""
    try:
        dt = datetime.fromtimestamp(timestamp)
        now = datetime.now()
        diff = now - dt
        
        # Relative time for recent files
        if diff.days == 0:
            if diff.seconds < 60:
                return "Just now"
            elif diff.seconds < 3600:
                minutes = diff.seconds // 60
                return f"{minutes} minute{'s' if minutes != 1 else ''} ago"
            else:
                hours = diff.seconds // 3600
                return f"{hours} hour{'s' if hours != 1 else ''} ago"
        elif diff.days == 1:
            return "Yesterday"
        elif diff.days < 7:
            return f"{diff.days} days ago"
        
        # Absolute date for older files
        if dt.year == now.year:
            return dt.strftime("%b %d")
        return dt.strftime("%b %d, %Y")
    except (ValueError, OSError):
        return "Unknown"

In [ ]:
import time

# Test format_timestamp with current time
now = time.time()
assert format_timestamp(now) == "Just now"
assert format_timestamp(now - 120) == "2 minutes ago"  # 2 minutes ago
assert format_timestamp(now - 7200) == "2 hours ago"   # 2 hours ago

# Test invalid timestamp
assert format_timestamp(-999999999999999) == "Unknown"

print("format_timestamp tests passed!")

format_timestamp tests passed!


## Pattern Matching

In [ ]:
#| export
def matches_glob_patterns(
    path: str,  # File path to check
    patterns: List[str]  # List of glob patterns to match against
) -> bool:  # True if path matches any pattern
    """Check if path matches any of the glob patterns."""
    if not patterns:
        return False
    
    for pattern in patterns:
        if fnmatch.fnmatch(path, pattern):
            return True
        # Also check just the filename
        if '/' in path or '\\' in path:
            filename = path.replace('\\', '/').split('/')[-1]
            if fnmatch.fnmatch(filename, pattern):
                return True
    return False

In [ ]:
# Test matches_glob_patterns
assert matches_glob_patterns("/path/to/file.py", ["*.py"]) == True
assert matches_glob_patterns("/path/to/file.py", ["*.txt"]) == False
assert matches_glob_patterns("/path/to/file.py", ["*.txt", "*.py"]) == True
assert matches_glob_patterns("file.py", ["*.py"]) == True
assert matches_glob_patterns("/path/to/.hidden", [".*"]) == True
assert matches_glob_patterns("/path/to/file.py", []) == False

print("matches_glob_patterns tests passed!")

matches_glob_patterns tests passed!


## MIME Type Detection

In [ ]:
#| export
def get_mime_type(
    path: str  # File path to check
) -> Optional[str]:  # MIME type string or None if unknown
    """Determine MIME type for a file based on extension."""
    mime_type, _ = mimetypes.guess_type(path)
    return mime_type

In [ ]:
# Test get_mime_type
assert get_mime_type("file.py") == "text/x-python"
assert get_mime_type("file.json") == "application/json"
assert get_mime_type("file.mp3") == "audio/mpeg"
assert get_mime_type("file.mp4") == "video/mp4"
assert get_mime_type("file.png") == "image/png"
assert get_mime_type("file.unknown_extension") is None

print("get_mime_type tests passed!")

get_mime_type tests passed!


In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()